In [6]:
import sys
import os
sys.path.append(os.path.abspath(r'e:\Sanjaya\MyProjects\Research\Multi Agent\Multi-Agent-Initialization'))

# Structured Web Data Extraction Agent - Version 3
This agent pipeline handles extracting complex thread structures using LangGraph.
It features **Inline Evaluations** (Context Relevance) and **Autonomous Recovery** by hooking into the `replan_flag`.

In [7]:
import os
import json
import warnings
import re
import requests
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

from typing import Annotated, Literal, Optional, List, Dict, Any
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, START, StateGraph, END
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

import prompts

# Setup LLMs
reasoning_llm = ChatGroq(model='llama-3.1-8b-instant')
llm = reasoning_llm

# Specialized State for Data Extracting Pipeline
class State(MessagesState):
    enabled_agents: Optional[List[str]]
    plan: Optional[Dict[str, Dict[str, Any]]]
    user_query: Optional[str]
    current_step: int
    replan_flag: Optional[bool]
    last_reason: Optional[str]
    replan_attempts: Optional[Dict[int, int]]
    agent_query: Optional[str]
    final_answer: Optional[str]
    context_relevance_score: Optional[float]

MAX_REPLANS = 2

In [8]:
tavily_tool = TavilySearchResults(max_results=5, search_depth='advanced', include_raw_content=True)

web_search_agent = create_react_agent(
    llm,
    tools=[tavily_tool],
    prompt=prompts.agent_system_prompt('You are the Web Data Extraction Researcher.')
)

def planner_node(state: State) -> Command[Literal['executor']]:
    plan_instructions = """You are the Planner. Determine a clear plan to fulfill the user query.
Generate a JSON output mapping each step "1", "2", etc., to an object with:
"agent": (web_researcher or synthesizer)
"pre_conditions": [list of strings]
"post_conditions": [list of strings]
"goal": "string"

User Query: """ + state.get("user_query", "")

    llm_reply = reasoning_llm.invoke([HumanMessage(content=plan_instructions)])
    
    try:
        content_str = llm_reply.content if isinstance(llm_reply.content, str) else str(llm_reply.content)
        content_str = content_str.replace('```json', '').replace('```', '')
        parsed_plan = json.loads(content_str)
    except Exception as e:
        parsed_plan = {
            "1": {"agent": "web_researcher"},
            "2": {"agent": "synthesizer"}
        }

    return Command(
        update={
            "plan": parsed_plan,
            "messages": [HumanMessage(content=json.dumps(parsed_plan), name="initial_plan")],
            "user_query": state.get("user_query", state.get("messages", [{}])[0].content if state.get("messages") else ""),
            "current_step": 1,
            "replan_flag": False,
            "last_reason": "",
        },
        goto="executor",
    )

def executor_node(state: State) -> Command[Literal["web_researcher", "synthesizer", "planner"]]:
    plan = state.get("plan", {})
    step = state.get("current_step", 1)
    
    # IDEA 3: Hook the inline evaluation results to `replan_flag` so we autonomously recover!
    replan_flag = state.get("replan_flag", False)
    if replan_flag:
        print("EXECUTOR DETECTED A REPLAN FLAG! Context relevance was too low. Autonomously replanning and retrying scraping...")
        return Command(
            update={
                "replan_flag": False,
                "messages": [HumanMessage(content="Low Context Relevance! Retrying.", name="executor")],
                "current_step": 1 # restart planning
            },
            goto="planner"
        )
    
    planned_agent = plan.get(str(step), {}).get("agent", "synthesizer")
    if planned_agent not in ["web_researcher", "synthesizer", "planner"]:
        planned_agent = "synthesizer"

    return Command(
        update={
            "messages": [HumanMessage(content=f"Routing to {planned_agent} for step {step}", name="executor")],
            "current_step": step + 1,
        },
        goto=planned_agent
    )

def evaluate_context_relevance(query: str, context: str) -> float:
    """Idea 1: Inline Evaluation of Context Relevance as an LLM Judge"""
    eval_prompt = f"""
    Evaluate the relevance of the extracted text to the main query.
    Is there substantive, useful content, or is it mostly empty/deleted/blocked?
    Return ONLY a float score between 0.0 (totally irrelevant/empty) and 1.0 (highly relevant/rich).
    
    Query: {query}
    Context excerpt: {context[:2000]}
    """
    try:
        reply = reasoning_llm.invoke([HumanMessage(content=eval_prompt)])
        score_str = reply.content.strip()
        # Find first float in response
        match = re.search(r'0\.\d+|1\.0', score_str)
        if match: return float(match.group())
        return 0.5
    except:
        return 0.5

def web_research_node(state: State) -> Command[Literal["executor"]]:
    query = state.get("user_query", "")
    print(f"WEB SEARCH POINTER -> Querying: {query[:50]}...")
    
    final_content = ""
    url_match = re.search(r"https?://[^\s]+", query)
    target_url = url_match.group(0) if url_match else None
    
    if target_url and "reddit.com" in target_url:
        try:
            json_url = target_url.rstrip("/") + ".json?limit=1000"
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(json_url, headers=headers)
            if response.status_code == 200:
                data = response.json()
                post_data = data[0]['data']['children'][0]['data']
                title = post_data.get('title', '')
                selftext = post_data.get('selftext', '')
                
                comments = []
                if len(data) > 1:
                    comment_children = data[1]['data'].get('children', [])
                    for child in comment_children:
                        if child.get('kind') == 't1':
                            c_data = child['data']
                            body = c_data.get('body', '').strip()
                            if body and body not in ["[deleted]", "[removed]"]:
                                comments.append(f"Comment (Score: {c_data.get('score', 0)}):\n{body}")
                
                combined_comments = "\n---\n".join(comments)
                if len(combined_comments) > 60000:
                    combined_comments = combined_comments[:60000]
                    
                final_content = f"Extracted from Reddit URL:\n\nTitle: {title}\n\nPost Content:\n{selftext}\n\nAll Comments:\n{combined_comments}"
        except Exception as e:
            print(f"Reddit JSON fetch failed: {e}")
    
    if not final_content:
        # FAKE AN EMPTY RESPONSE ON FIRST TRY if we want to demonstrate the retry mechanism
        # But here we'll let it happen naturally if no text is found
        try:
            agent_result = web_search_agent.invoke({"messages": [HumanMessage(content=query)]})
            final_content = agent_result["messages"][-1].content
        except Exception as e:
            final_content = f"Failed: {e}"
        
    # --- Idea 1: Evaluate Relevance ---
    relevance_score = evaluate_context_relevance(query, final_content)
    print(f"🛡️ [Inline Evaluation] Context Relevance Score: {relevance_score}")
    
    # --- Idea 3: Trigger Replan on low score ---
    trigger_replan = relevance_score < 0.4
    if trigger_replan:
        print("⚠️ Context relevance is too low! Flagging for Replan!")
        
    return Command(
        update={
            "messages": [HumanMessage(content=final_content, name="web_researcher")],
            "context_relevance_score": relevance_score,
            "replan_flag": trigger_replan
        },
        goto="executor"
    )

def synthesizer_node(state: State) -> Command[Literal[END]]:
    relevant_msgs = [m.content for m in state.get("messages", []) if getattr(m, "name", None) == "web_researcher"]
    prompt = f"User question: {state.get('user_query', '')}\n\nContext:\n\n" + "\n\n".join(relevant_msgs)
    
    if len(prompt) > 80000: prompt = prompt[:80000]
       
    llm_reply = reasoning_llm.invoke([HumanMessage(content=prompt)])
    answer = llm_reply.content if isinstance(llm_reply.content, str) else str(llm_reply.content)
    return Command(update={"final_answer": answer.strip(), "messages": [HumanMessage(content=answer.strip(), name="synthesizer")]}, goto=END)

In [9]:
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(State)
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("web_researcher", web_research_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.add_edge(START, "planner")

memory = MemorySaver()
graph = workflow.compile(checkpointer=memory) # Complete graph, no interrupt for V3 full run

In [10]:
target_url = "https://www.reddit.com/r/ClaudeCode/comments/1r5gk1d/40_days_of_vibe_coding_taught_me_the_most/"

query = f"Please thoroughly extract the main points, insights, and key takeaways from the following Reddit post URL:\n{target_url}"

state = {
    "messages": [HumanMessage(content=query)],
    "user_query": query,
}

config = {"configurable": {"thread_id": "v3_autonomous_run"}}

print("🚀 Starting V3 Autonomous Pipeline (Features Inline Eval & Replanning)...")
result = graph.invoke(state, config=config)

print("\n--- 📝 V3 FINAL SYNTHESIZED REPORT ---\n")
print(result.get("final_answer"))

🚀 Starting V3 Autonomous Pipeline (Features Inline Eval & Replanning)...
WEB SEARCH POINTER -> Querying: Please thoroughly extract the main points, insight...
🛡️ [Inline Evaluation] Context Relevance Score: 0.6

--- 📝 V3 FINAL SYNTHESIZED REPORT ---

However, I'm unable to access external links. But I can guide you through the process of analyzing the Reddit post and extracting the main points, insights, and key takeaways based on the provided context.

**Title:** "40 days of vibe coding taught me the most"

**Context:** The author faced an error code 413 while using the LLaMA model, indicating that the request was too large and exceeded the token limit.

**Possible Key Takeaways:**

1. **Token Limitations**: The author learned that there are token limitations when using the LLaMA model, specifically a 6000-token limit per minute.
2. **Request Size**: The author realized the importance of keeping the request size within the allowed limit to avoid errors and rate limit exceeded issues.
